# 👟 YOLOv8n-Pose (4 Coarse Keypoints) Two-Stage Training Pipeline on Google Colab
### Stage 1: Pretraining on `shoes_v2` (v11) → Stage 2: Fine-Tuning on `shuffled_v3` (v12)

This notebook provides the complete end-to-end workflow to download datasets directly from Roboflow, clean & remap keypoints, train YOLOv8n-pose on 4 coarse keypoints per foot, export production ONNX models, and benchmark on test video.

---
### 📌 Execution Pipeline:
1. **Step 1:** Mount Google Drive (for checkpoints and model storage).
2. **Step 2:** Clone GitHub repository (`yolo_stage1` branch) & install dependencies (`ultralytics`, `roboflow`, `onnx`, etc.).
3. **Step 3:** Download V2 (v11) & V3 (v12) from Roboflow → Remap, Repair & Shuffle → Convert to 4-KP.
4. **Step 4:** **Stage 1 Pretraining:** Train `yolov8n-pose.pt` on `data/shoes_v2_4kp` (100 epochs).
5. **Step 5:** **Stage 2 Fine-Tuning:** Fine-tune Stage 1 best weights on `data/shuffled_v3_4kp` (150 epochs).
6. **Step 6:** Validate final model on test split.
7. **Step 7:** Export to ONNX (FP32, FP16, INT8) & synchronize with Drive.
8. **Step 8:** Run video inference on `test_video.mp4` with 4-KP HUD overlay.
9. **Step 9:** Download final models to local machine.

### Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create experiment directories in Google Drive
DRIVE_DIR = '/content/drive/MyDrive/Shoes_VTO_Experiments'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f"✅ Google Drive Mounted! Permanent storage directory: {DRIVE_DIR}")

### Step 2: Clone GitHub Repository & Install Dependencies

In [ ]:
%cd /content
!rm -rf Shoes_VTO
!git clone -b yolo_stage1 https://github.com/HagAli22/Shoes_VTO.git
%cd Shoes_VTO

# Install dependencies
!pip install -q ultralytics roboflow albumentations onnx onnxruntime onnxsim onnxconverter-common opencv-python-headless tqdm

import torch, ultralytics
print(f"\nPyTorch Version     : {torch.__version__}")
print(f"Ultralytics Version : {ultralytics.__version__}")
print(f"CUDA Available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device          : {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### Step 3: Download Datasets from Roboflow (V2 & V3), Clean, Remap & Convert to 4-KP Format
*(Enter your Roboflow API key below to automatically download version 11 and version 12)*

In [ ]:
import os, shutil
from pathlib import Path

# 🔑 Put your Roboflow API key here
ROBOFLOW_API_KEY = "YOUR_ROBOFLOW_API_KEY"

drive_v2_cache = f'{DRIVE_DIR}/shoes_v2_4kp.zip'
drive_v3_cache = f'{DRIVE_DIR}/shuffled_v3_4kp.zip'

# Check if cached 4-KP datasets already exist in Drive
if os.path.exists(drive_v2_cache) and os.path.exists(drive_v3_cache):
    print("📦 Found cached 4-KP datasets in Google Drive! Extracting...")
    !unzip -q -o "{drive_v2_cache}" -d data/
    !unzip -q -o "{drive_v3_cache}" -d data/
    !python tools/convert_dataset_to_4kp.py --all
    print("✅ Datasets restored from Drive cache!")
else:
    print("⬇️ Downloading datasets from Roboflow...")
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("shoes-vto").project("fingers_keypoint")
    
    # 1. Download V2 (Version 11)
    print("\n[1/4] Downloading V2 (Version 11)...")
    dataset_v2 = project.version(11).download("yolov8", location="data/shoes_v2_raw")
    
    # 2. Download V3 (Version 12)
    print("\n[2/4] Downloading V3 (Version 12)...")
    dataset_v3 = project.version(12).download("yolov8", location="data/shoes_v3_raw")
    
    # 3. Clean, repair coordinates, and remap classes (0<->1)
    print("\n[3/4] Preparing and shuffling V2 & V3 datasets...")
    !python tools/prepare_shuffled_dataset.py --source data/shoes_v2_raw --dest data/shoes_v2 --remap
    !python tools/prepare_shuffled_dataset.py --source data/shoes_v3_raw --dest data/shuffled_v3 --remap
    
    # 4. Convert to 4-KP Native Format
    print("\n[4/4] Converting datasets to 4 coarse keypoints format...")
    !python tools/convert_dataset_to_4kp.py --all
    
    # Cache to Google Drive for future instant runs
    print("\n💾 Caching 4-KP datasets to Google Drive...")
    !zip -q -r "{drive_v2_cache}" data/shoes_v2_4kp
    !zip -q -r "{drive_v3_cache}" data/shuffled_v3_4kp
    print(f"✅ Datasets cached to Google Drive: {drive_v2_cache} and {drive_v3_cache}")

# Display Summary
print("\n=======================================================")
print("          4-KP DATASETS READY FOR TRAINING             ")
print("=======================================================")
for ds in ["shoes_v2_4kp", "shuffled_v3_4kp"]:
    print(f"\nDataset: {ds}")
    for split in ["train", "valid", "test"]:
        p = f"data/{ds}/{split}/images"
        if os.path.exists(p):
            print(f"  {split:5s} images: {len(os.listdir(p))}")

### Step 4: Stage 1 Pretraining on `shoes_v2_4kp`
*(Trains pretrained `yolov8n-pose.pt` on the coarse dataset for initial weight convergence)*

In [ ]:
!yolo pose train \
  data=configs/shoes_v2_4kp.yaml \
  model=yolov8n-pose.pt \
  epochs=100 \
  imgsz=320 \
  batch=32 \
  device=0 \
  optimizer=AdamW \
  lr0=0.002 \
  patience=25 \
  project=outputs/stage1_4kp \
  name=pretrain_shoes_v2

# Backup Stage 1 best checkpoint to Google Drive
!cp outputs/stage1_4kp/pretrain_shoes_v2/weights/best.pt "{DRIVE_DIR}/checkpoints/yolo_4kp_stage1_pretrain.pt"
print(f"\n✅ Stage 1 checkpoint saved to: {DRIVE_DIR}/checkpoints/yolo_4kp_stage1_pretrain.pt")

### Step 5: Stage 2 Fine-Tuning on `shuffled_v3_4kp`
*(Fine-tunes Stage 1 weights on the clean, uniformly balanced 1,102-image dataset)*

In [ ]:
!yolo pose train \
  data=configs/shuffled_v3_4kp.yaml \
  model=outputs/stage1_4kp/pretrain_shoes_v2/weights/best.pt \
  epochs=150 \
  imgsz=320 \
  batch=32 \
  device=0 \
  optimizer=AdamW \
  lr0=0.001 \
  patience=35 \
  project=outputs/stage1_4kp \
  name=finetune_shuffled_v3

# Backup final fine-tuned checkpoint to Google Drive
!cp outputs/stage1_4kp/finetune_shuffled_v3/weights/best.pt "{DRIVE_DIR}/checkpoints/yolo_4kp_best.pt"
print(f"\n✅ Stage 2 fine-tuned checkpoint saved to: {DRIVE_DIR}/checkpoints/yolo_4kp_best.pt")

### Step 6: Validate Final Model on Test Split

In [ ]:
!yolo pose val \
  data=configs/shuffled_v3_4kp.yaml \
  model=outputs/stage1_4kp/finetune_shuffled_v3/weights/best.pt \
  imgsz=320 \
  split=test \
  device=0

### Step 7: Export to ONNX (FP32, FP16, INT8) & Synchronize with Drive

In [ ]:
!python -m src.export.export_yolo_4kp \
  --weights outputs/stage1_4kp/finetune_shuffled_v3/weights/best.pt \
  --output_dir deliverables/stage_a_4kp \
  --prefix stage-a-320-yolo4kp

# Copy exported models to Google Drive
!cp -r deliverables/stage_a_4kp/* "{DRIVE_DIR}/models/"
print(f"\n✅ Exported ONNX models synchronized to Google Drive: {DRIVE_DIR}/models/")
!ls -lh "{DRIVE_DIR}/models/"

### Step 8: Test Video Inference & Visual Benchmark (`test_video.mp4`)
*(Generates video with bounding boxes, 4-KP skeleton overlay, and real-time latency HUD)*

In [ ]:
import os

# Search candidate video locations
video_candidates = [
    f'{DRIVE_DIR}/test_video.mp4',
    'deliverables/stage_a_16kp/test_video.mp4',
    'data/test_video.mp4'
]

video_path = next((vc for vc in video_candidates if os.path.exists(vc)), None)

if video_path is None:
    print("❌ test_video.mp4 not found. Please upload test_video.mp4 to Google Drive at:", f'{DRIVE_DIR}/test_video.mp4')
else:
    print(f"🎥 Running video evaluation on: {video_path}")
    out_video_drive = f'{DRIVE_DIR}/yolo_4kp_annotated_video.mp4'
    
    !python tools/evaluation/eval_yolo_4kp_video.py \
      --video "{video_path}" \
      --model outputs/stage1_4kp/finetune_shuffled_v3/weights/best.pt \
      --output "{out_video_drive}" \
      --conf 0.25 \
      --device 0
    
    print(f"\n✅ Annotated output video saved directly to Google Drive: {out_video_drive}")

### Step 9: Download Artifacts to Local PC (Optional)

In [ ]:
from google.colab import files

# Download the FP32 and FP16 models + best checkpoint
try:
    files.download('deliverables/stage_a_4kp/stage-a-320-yolo4kp-fp32.onnx')
    files.download('deliverables/stage_a_4kp/stage-a-320-yolo4kp-fp16.onnx')
    files.download('outputs/stage1_4kp/finetune_shuffled_v3/weights/best.pt')
except Exception as e:
    print(f"Note: {e}")